# AlexandriaX Subtask 1 Baseline — Kaggle T4 / Unsloth Adaptation

This is the official AlexandriaX Subtask 1 baseline (context-aware English-to-
dialectal-Arabic dialogue translation), adapted to fine-tune and run on a
single Kaggle **Tesla T4 (16GB)** instead of the original A100 (80GB). It's
meant for anyone entering the shared task without institutional compute
access — everything here runs on the free GPU tier Kaggle provides.

**Original baseline (A100, plain HF + PEFT):**
https://colab.research.google.com/drive/1n5OFTrvCdZFqE9qttyB0FMMfrL3GhVbm
(linked from the [official task page](https://alexandriax.dlnlp.ai/#subtasks))

## What's different from the original, and why

| Aspect | Original baseline (A100) | This notebook (Kaggle T4) |
|---|---|---|
| Model loading / LoRA | Plain `transformers` + `peft.LoraConfig`, `target_modules="all-linear"` | [Unsloth](https://github.com/unslothai/unsloth) `FastLanguageModel` — 4-bit loading and LoRA fused into one call, explicit `target_modules` list (attention + MLP projections), much lower VRAM footprint |
| Gradient checkpointing | Native HF (`gradient_checkpointing=True`) | Unsloth's own checkpointing (`use_gradient_checkpointing="unsloth"`), with HF-side checkpointing turned off to avoid the two conflicting |
| Train batch size | 32 (fits an A100's 80GB) | 8, same `gradient_accumulation_steps=4` — a T4's 16GB can't hold a batch of 32 even in 4-bit, so the effective batch size drops from 128 to 32. **This is the one change that can affect results, not just runtime** — worth re-tuning if you have the compute to spare. |
| Precision | bf16 if supported, else fp16 (A100 supports bf16) | fp16 only (T4 has no bf16 support) |
| Dev data source | Local CodaBench-downloaded files (`development_data/`) + a separate `references.jsonl` lookup | The HF `dev` split directly (`load_dataset(..., split="dev")`) — simpler when you don't have the CodaBench files locally, at the cost of not exercising the local-file-loading path the real submission pipeline uses |
| Experiment tracking | `report_to="none"` | Weights & Biases (`report_to="wandb"`), with checkpoints logged as W&B artifacts |
| Checkpoint reload for inference | Reloaded from a local `CHECKPOINT_DIR` (same session) | Downloaded from a **W&B artifact** — Kaggle sessions are ephemeral and time-limited, so training and inference often happen in separate sessions with no shared disk |
| Crash recovery | Not needed (single long-running A100 job) | A `trainer.train(resume_from_checkpoint=...)` cell that resumes from the last W&B checkpoint artifact, for when a Kaggle session times out mid-training |
| `save_steps` / `eval_steps` / `save_total_limit` | 200 / 200 / 2 | 500 / 500 / 5 |
| Submission zip generation | Included (writes a CodaBench-shaped dev submission zip) | Not included here — this notebook stops at scoring; see `models/infer.py --make-submission-zip` for that step |

Two fixes made while cleaning this up (both silent failure modes in the
original working notebook):
- **`MAX_SEQ_LENGTH` was only defined in the training section**, but the
  inference section's `generate_translations` reads it — if you started a
  fresh Kaggle session and jumped straight to inference (the whole point of
  loading a checkpoint from a W&B artifact), this would raise a `NameError`.
  Now defined in both sections.
- The tokenizer padding fix checked `if tokenizer.pad_token is None`, but
  Unsloth's `FastLanguageModel` loader never returns `None` for NileChat-3B's
  tokenizer — it defaults to `'<|vision_pad|>'`, a placeholder token the
  model was never trained to treat as padding. The check now specifically
  looks for that value.

Everything else (prompt format, data normalization, scoring) is unchanged
from the official baseline — only the compute-constrained training/inference
path differs.

## Kaggle environment setup

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # single T4
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"  # reduces fragmentation OOMs on 16GB

## Installs

Comment out the `unsloth`/`trl` install lines if you're running an
inference-only session (loading a checkpoint from W&B) and don't need to
re-run training — they're the slowest part of the install.

In [ ]:
!pip install unsloth  # not needed for inference-only sessions
!pip install trl      # not needed for inference-only sessions
!pip install sacrebleu sentencepiece
!pip install -U bitsandbytes>=0.46.1

## Authentication (Kaggle Secrets)

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    print("Hugging Face Hub authenticated successfully via environment variable!")
except Exception:
    print("Could not find your Kaggle Secret. Make sure Add-ons -> Secrets has 'HF_TOKEN' active.")

In [ ]:
import wandb
import os

try:
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = wandb_key
    wandb.login()
except Exception:
    print("Could not find your Kaggle Secret! Make sure Add-ons -> Secrets has 'WANDB_API_KEY' active.")

## Imports & config

In [ ]:
import os
import json
import random
import torch
from datasets import Dataset, get_dataset_config_names, get_dataset_split_names, load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

os.environ["WANDB_ENTITY"] = "RosettaAtAlexandriaX"
os.environ["WANDB_PROJECT"] = "DialectalArabicMT"
os.environ["WANDB_LOG_MODEL"] = "checkpoint"


In [ ]:
OUTPUT_ROOT = Path("outputs")
CHECKPOINT_DIR = OUTPUT_ROOT / "nilechat_alexandriaX_lora"
EVAL_DIR = OUTPUT_ROOT / "evaluation"
SCORE_DIR = OUTPUT_ROOT / "scores"

DATASET_NAME = "UBC-NLP/alexandria"
COUNTRIES = ["EG", "JO", "LB", "LY", "MA", "MR", "OM", "PS", "SA", "SD", "SY", "TN", "YE"]

SEED = 42
random.seed(SEED)

## Load data

Same prompt format and data normalization as the official baseline. The one
difference: development data is loaded directly from the HF `dev` split
(`load_hf_dev_records`) rather than from local CodaBench-downloaded files —
see the comparison table above.

In [ ]:
def discover_hf_splits(dataset_name: str, countries: list[str]) -> dict[str, set[str]]:
    available_configs = set(get_dataset_config_names(dataset_name))
    split_map: dict[str, set[str]] = {}

    for country in countries:
        if country not in available_configs:
            split_map[country] = set()
            continue
        split_map[country] = set(get_dataset_split_names(dataset_name, country))

    return split_map


hf_split_map = discover_hf_splits(DATASET_NAME, COUNTRIES)
TRAIN_COUNTRIES = [country for country in COUNTRIES if "train" in hf_split_map.get(country, set())]

print("Hugging Face train countries:", TRAIN_COUNTRIES)
print("Countries without HF train split:", [c for c in COUNTRIES if c not in TRAIN_COUNTRIES])

In [ ]:
def _turn_order(turn: dict, fallback: int = 0) -> int:
    try:
        return int(turn.get("turn_order", fallback))
    except (TypeError, ValueError):
        return fallback


def sorted_turns(turns: list[dict]) -> list[dict]:
    return sorted(turns or [], key=lambda turn: _turn_order(turn))


def read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def normalize_hf_record(row: dict) -> dict:
    english_turns = sorted_turns(row.get("english_conversation", []))
    dialect_turns = {
        _turn_order(turn, index + 1): turn
        for index, turn in enumerate(sorted_turns(row.get("dialectal_conversation", [])))
    }

    turns = []
    whole_conversation_lines = []
    for index, english_turn in enumerate(english_turns, start=1):
        order = _turn_order(english_turn, index)
        reference_turn = dialect_turns.get(order, {})
        speaker = str(english_turn.get("speaker", "")).strip()
        sentence = str(english_turn.get("text", "")).strip()
        whole_conversation_lines.append(f"{speaker}: {sentence}" if speaker else sentence)
        turns.append({
            "turn_order": order,
            "speaker": speaker,
            "sentence": sentence,
            "direction": str(english_turn.get("direction", "")).strip(),
            "reference": str(reference_turn.get("text", "")).strip(),
        })

    return {
        "conv_id": str(row.get("conv_id", "")).strip(),
        "country": str(row.get("country", "")).strip(),
        "domain": str(row.get("domain", "")).strip(),
        "dialect": str(row.get("dialect", "Arabic Dialect")).strip() or "Arabic Dialect",
        "participants": str(row.get("participants", "")).strip(),
        "whole_conversation": "\n\n".join(whole_conversation_lines),
        "turns": turns,
    }


def format_history(history: list[dict]) -> str:
    if not history:
        return "No previous turns (Start of conversation)."

    lines = []
    for item in history:
        speaker = item.get("speaker", "Speaker") or "Speaker"
        lines.append(f"{speaker}: {item['sentence']}\nTranslation: {item['translation']}")
    return "\n".join(lines)


def build_prompt(record: dict, turn: dict, history: list[dict]) -> str:
    dialect = record.get("dialect") or "Arabic Dialect"
    domain = record.get("domain") or "Unknown Domain"
    participants = record.get("participants") or "Unknown Participants"
    direction = turn.get("direction") or "Unknown"
    speaker = turn.get("speaker") or "Unknown Speaker"

    return (
        f"You are an expert translator. Translate the English sentence into {dialect}.\n\n"
        f"### Metadata:\n"
        f"- Country: {record.get('country', '')}\n"
        f"- Domain: {domain}\n"
        f"- Participants: {participants}\n"
        f"- Speaker: {speaker}\n"
        f"- Speaker Direction: {direction}\n\n"
        f"### Conversation History:\n"
        f"{format_history(history)}\n\n"
        f"### Sentence to Translate:\n"
        f"{turn.get('sentence', '').strip()}\n\n"
        f"### Translation:\n"
    )


def create_finetuning_pairs(record: dict) -> list[dict]:
    pairs = []
    history: list[dict] = []

    for turn in sorted_turns(record.get("turns", [])):
        sentence = turn.get("sentence", "").strip()
        reference = turn.get("reference", "").strip()
        if not sentence or not reference:
            continue

        pairs.append({
            "prompt": build_prompt(record, turn, history),
            "response": reference,
            "country": record.get("country", ""),
            "conv_id": record.get("conv_id", ""),
            "turn_order": turn.get("turn_order"),
        })
        history.append({
            "speaker": turn.get("speaker", ""),
            "sentence": sentence,
            "translation": reference,
        })

    return pairs

In [ ]:
def load_hf_train_records(countries: list[str]) -> list[dict]:
    records: list[dict] = []

    for country in tqdm(countries, desc="Loading HF train countries", unit="country"):
        dataset = load_dataset(DATASET_NAME, country, split="train")
        country_records = [normalize_hf_record(row) for row in dataset]
        records.extend(country_records)
        print(f"Loaded {len(country_records):>5} train conversations for {country}")

    return records


def load_hf_dev_records(countries: list[str]) -> list[dict]:
    records: list[dict] = []

    for country in tqdm(countries, desc="Loading HF dev countries", unit="country"):
        dataset = load_dataset(DATASET_NAME, country, split="dev")
        country_records = [normalize_hf_record(row) for row in dataset]
        records.extend(country_records)
        print(f"Loaded {len(country_records):>5} dev conversations for {country}")

    return records

Only needed for a training session — comment out if you're loading a checkpoint straight from W&B for inference.

In [ ]:
hf_train_records = load_hf_train_records(TRAIN_COUNTRIES)
train_pairs = [pair for record in hf_train_records for pair in create_finetuning_pairs(record)]
print("Train conversations:", len(hf_train_records))
print("Train turns:", len(train_pairs))

development_records = load_hf_dev_records(TRAIN_COUNTRIES)
development_pairs = [pair for record in development_records for pair in create_finetuning_pairs(record)]
print("Development conversations:", len(development_records))
print("Development turns:", len(development_pairs))

## Training

Hyperparameters, relative to the original A100 baseline: `BATCH_SIZE` is
32 -> 8 (a T4's 16GB can't fit a batch of 32 in memory even in 4-bit —
`GRADIENT_ACCUMULATION_STEPS` stays at 4, so effective batch size drops from
128 to 32). `LORA_R` / `LORA_ALPHA` / `LORA_DROPOUT` are unchanged.

In [ ]:
MODEL_NAME = "UBC-NLP/NileChat-3B-Base"
NUM_EPOCHS = 1

MAX_SEQ_LENGTH = 1024
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4

USE_4BIT = True
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

Loaded via Unsloth's `FastLanguageModel` instead of plain
`transformers.AutoModelForCausalLM` + `peft.prepare_model_for_kbit_training`
— 4-bit loading and LoRA-readiness happen in one call, with a much smaller
memory footprint than the equivalent HF/PEFT path.

**Fix applied here:** NileChat-3B's tokenizer, loaded through Unsloth,
defaults its pad token to `'<|vision_pad|>'` rather than `None` — the
original baseline's `if tokenizer.pad_token is None` check never fires for
this loading path, silently leaving the wrong pad token in place. Checking
for `'<|vision_pad|>'` specifically fixes it.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=USE_4BIT,
    dtype=torch.float16,
)

if tokenizer.pad_token == "<|vision_pad|>":
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("EOS token:", tokenizer.eos_token)
print("PAD token:", tokenizer.pad_token)
print(f"Model dtype: {next(model.parameters()).dtype}")

In [ ]:
def build_sft_dataset(pairs: list[dict]) -> Dataset:
    rows = []
    for item in pairs:
        rows.append({
            "prompt": item["prompt"],
            "completion": item["response"].strip() + tokenizer.eos_token,
        })
    return Dataset.from_list(rows)


train_dataset = build_sft_dataset(train_pairs)
eval_dataset = build_sft_dataset(development_pairs)

Wrap with LoRA via Unsloth's `get_peft_model` (instead of passing a
`peft.LoraConfig` to `SFTTrainer`). `target_modules` is an explicit list here
rather than the original's `"all-linear"` string — Unsloth requires
naming the modules; this list covers the standard attention + MLP
projections but, unlike `"all-linear"`, won't pick up any additional linear
layers a model might have outside those (not applicable to NileChat-3B's
architecture, but worth knowing if you port this to a different base
model).

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized checkpointing
    random_state=SEED,
    use_rslora=False,
)

`gradient_checkpointing=False` here is intentional — Unsloth's
`get_peft_model` above already installed its own checkpointing, and turning
on HF's native one as well causes the two to conflict. `save_steps` /
`eval_steps` / `save_total_limit` are looser than the original (500/500/5 vs
200/200/2) since Kaggle sessions have a wall-clock time limit and frequent
checkpointing eats into it. Logging goes to Weights & Biases instead of
being disabled, so a crashed session can be resumed (see below).

In [ ]:
training_args = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    max_length=MAX_SEQ_LENGTH,

    logging_steps=10,
    save_steps=500,
    save_total_limit=5,
    eval_strategy="steps",
    eval_steps=500,
    bf16=False,   # T4 has no bf16 support
    fp16=True,

    completion_only_loss=True,
    gradient_checkpointing=False,  # handled by Unsloth's get_peft_model above

    optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
    report_to="wandb",
    remove_unused_columns=True,
    packing=False,

    run_name="CREATE-A-RUN-NAME",  # replace with a descriptive run name
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

trainer.model.save_pretrained(CHECKPOINT_DIR)
tokenizer.save_pretrained(CHECKPOINT_DIR)

print(f"Saved LoRA adapter and tokenizer to: {CHECKPOINT_DIR}")

## Resuming after a Kaggle session timeout

Kaggle sessions have a wall-clock limit, so a long training run can get cut
off mid-way. Since checkpoints are logged to W&B as artifacts
(`WANDB_LOG_MODEL="checkpoint"` above), a fresh session can pull the last one
and resume from it. Not needed on a single long-running A100 job, which is
why the original baseline doesn't have this cell.

In [ ]:
# Uncomment and fill in your checkpoint artifact path to resume:

# run = wandb.init(entity="RosettaAtAlexandriaX", project="DialectalArabicMT", job_type="resume_training")
# artifact_path = "RosettaAtAlexandriaX/DialectalArabicMT/YOUR-CHECKPOINT"  # "entity/project/artifact_name:version"
# artifact = run.use_artifact(artifact_path, type="model")
# artifact_dir = artifact.download()
# print(f"Checkpoint successfully downloaded to: {artifact_dir}")
# trainer.train(resume_from_checkpoint=artifact_dir)

## Inference with a saved checkpoint

This section is self-contained and can run in a fresh Kaggle session — it
downloads the LoRA adapter from a W&B artifact rather than assuming the
training section already ran in this session (the original baseline reloads
straight from a local `CHECKPOINT_DIR`, which only works within the same
session/disk).

`MAX_SEQ_LENGTH` is redefined here since this section is meant to run
independently of the training section above.

In [ ]:
MODEL_NAME = "UBC-NLP/NileChat-3B-Base"
MAX_SEQ_LENGTH = 1024
GENERATION_BATCH_SIZE = 128
MAX_NEW_TOKENS = 128
USE_4BIT = True

In [ ]:
run = wandb.init(entity="RosettaAtAlexandriaX", project="DialectalArabicMT", job_type="inference")
artifact_path = "RosettaAtAlexandriaX/DialectalArabicMT/model-CREATE-A-RUN-NAME:v1"  # "entity/project/artifact_name:version"
artifact = run.use_artifact(artifact_path, type="model")
artifact_dir = Path(artifact.download())

print(f"Checkpoint successfully downloaded to: {artifact_dir}")


def model_compute_dtype() -> torch.dtype:
    # fp16 on T4, even though the original baseline prefers bf16 when available.
    return torch.float16


def build_quantization_config() -> BitsAndBytesConfig | None:
    if not USE_4BIT:
        return None

    return BitsAndBytesConfig(
        load_in_4bit=USE_4BIT,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=model_compute_dtype(),
        bnb_4bit_use_double_quant=True,
    )


adapter_config_path = artifact_dir / "adapter_config.json"
if not adapter_config_path.exists():
    raise FileNotFoundError(
        f"No LoRA adapter found at {artifact_dir}. Run the training section first, "
        "or point artifact_path at a completed checkpoint."
    )

tokenizer_source = artifact_dir if (artifact_dir / "tokenizer_config.json").exists() else MODEL_NAME
tokenizer = AutoTokenizer.from_pretrained(tokenizer_source, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=build_quantization_config(),
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, artifact_dir)
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()

model.eval()
model.config.use_cache = True

print(f"Loaded fine-tuned model from: {artifact_dir}")

## Prediction and scoring helpers

Same context-aware, turn-by-turn generation as the official baseline: later
turns in a conversation use the model's own predictions for earlier turns as
history, not the gold references.

In [ ]:
def count_turns(records: list[dict]) -> int:
    return sum(len(record.get("turns", [])) for record in records)


def chunks(items: list, batch_size: int):
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


def clean_generation(text: str) -> str:
    text = text.strip()
    for marker in ("\n###", "### Sentence to Translate:", "### Translation:"):
        if marker in text:
            text = text.split(marker, 1)[0].strip()
    return text


def generate_translations(prompts: list[str], model, tokenizer, max_new_tokens: int = MAX_NEW_TOKENS) -> list[str]:
    previous_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_tokens = outputs[:, inputs["input_ids"].shape[1]:]
    decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    tokenizer.padding_side = previous_padding_side
    return [clean_generation(text) for text in decoded]


def generate_prediction_records(
    records: list[dict],
    model,
    tokenizer,
    batch_size: int = GENERATION_BATCH_SIZE,
    desc: str = "Generating predictions",
) -> list[dict]:
    histories: dict[int, list[dict]] = defaultdict(list)
    outputs = [
        {"conv_id": record["conv_id"], "country": record["country"], "turns": []}
        for record in records
    ]
    max_turns = max((len(record.get("turns", [])) for record in records), default=0)

    with tqdm(total=count_turns(records), desc=desc, unit="turn") as progress:
        for turn_position in range(max_turns):
            active = []
            for record_index, record in enumerate(records):
                turns = sorted_turns(record.get("turns", []))
                if turn_position >= len(turns):
                    continue
                turn = turns[turn_position]
                prompt = build_prompt(record, turn, histories[record_index])
                active.append((record_index, turn, prompt))

            for batch in chunks(active, batch_size):
                prompts = [item[2] for item in batch]
                translations = generate_translations(prompts, model, tokenizer)
                for (record_index, turn, _), translation in zip(batch, translations):
                    outputs[record_index]["turns"].append({
                        "turn_order": int(turn["turn_order"]),
                        "prediction": translation,
                    })
                    histories[record_index].append({
                        "speaker": turn.get("speaker", ""),
                        "sentence": turn.get("sentence", ""),
                        "translation": translation,
                    })
                progress.update(len(batch))

    return outputs


def write_jsonl(records: list[dict], path: Path, desc: str | None = None) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    iterator = tqdm(records, desc=desc or f"Writing {path.name}", unit="conversation")
    with path.open("w", encoding="utf-8") as handle:
        for record in iterator:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    return path

In [ ]:
def prediction_map(prediction_records: list[dict]) -> dict[tuple[str, str, int], str]:
    mapped = {}
    for record in prediction_records:
        country = str(record.get("country", "")).strip()
        conv_id = str(record.get("conv_id", "")).strip()
        for index, turn in enumerate(record.get("turns", []), start=1):
            turn_order = int(turn.get("turn_order", index))
            key = (country, conv_id, turn_order)
            if key in mapped:
                raise ValueError(f"Duplicate prediction key: {key}")
            mapped[key] = str(turn.get("prediction", "")).strip()
    return mapped


def reference_map(reference_records: list[dict]) -> dict[tuple[str, str, int], str]:
    mapped = {}
    for record in reference_records:
        country = str(record.get("country", "")).strip()
        conv_id = str(record.get("conv_id", "")).strip()
        for index, turn in enumerate(sorted_turns(record.get("turns", [])), start=1):
            reference = str(turn.get("reference", "")).strip()
            if not reference:
                continue
            turn_order = int(turn.get("turn_order", index))
            key = (country, conv_id, turn_order)
            if key in mapped:
                raise ValueError(f"Duplicate reference key: {key}")
            mapped[key] = reference
    return mapped


def score_prediction_records(
    prediction_records: list[dict],
    reference_records: list[dict],
    output_path: Path | None = None,
    desc: str = "Scoring countries",
) -> dict:
    try:
        from sacrebleu.metrics import BLEU, CHRF
    except ImportError as exc:
        raise ImportError("Install scoring dependencies with: pip install sacrebleu sentencepiece") from exc

    predictions = prediction_map(prediction_records)
    references = reference_map(reference_records)

    missing = sorted(set(references) - set(predictions))
    extra = sorted(set(predictions) - set(references))
    if missing or extra:
        raise ValueError(f"Prediction/reference mismatch. Missing={missing[:5]}, extra={extra[:5]}")

    try:
        bleu = BLEU(tokenize="flores200", effective_order=False)
    except Exception as exc:
        raise RuntimeError(
            "Could not initialize SacreBLEU's flores200 tokenizer. "
            "Install/upgrade sacrebleu and sentencepiece."
        ) from exc
    chrf = CHRF(word_order=2)

    by_country: dict[str, list[tuple[str, str]]] = defaultdict(list)
    for key in tqdm(sorted(references), desc="Aligning predictions", unit="turn"):
        country = key[0]
        by_country[country].append((predictions[key], references[key]))

    scores: dict[str, float | int] = {}
    spbleu_values = []
    chrfpp_values = []
    for country, rows in tqdm(sorted(by_country.items()), desc=desc, unit="country"):
        hypotheses = [prediction for prediction, _ in rows]
        refs = [reference for _, reference in rows]
        spbleu = bleu.corpus_score(hypotheses, [refs]).score
        chrfpp = chrf.corpus_score(hypotheses, [refs]).score
        spbleu_values.append(spbleu)
        chrfpp_values.append(chrfpp)
        scores[f"spbleu_{country}"] = round(spbleu, 6)
        scores[f"chrfpp_{country}"] = round(chrfpp, 6)

    scores["spbleu_avg"] = round(sum(spbleu_values) / len(spbleu_values), 6)
    scores["chrfpp_avg"] = round(sum(chrfpp_values) / len(chrfpp_values), 6)
    scores["num_countries"] = len(by_country)
    scores["num_turns"] = len(references)
    scores["num_conversations"] = len({key[:2] for key in references})

    if output_path is not None:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        output_path.write_text(json.dumps(scores, ensure_ascii=False, indent=2, sort_keys=True), encoding="utf-8")
        print(f"Scores saved to {output_path}")

    return scores


def display_scores(scores: dict) -> None:
    summary_keys = ["spbleu_avg", "chrfpp_avg", "num_countries", "num_conversations", "num_turns"]
    country_rows = []
    for key, value in scores.items():
        if not key.startswith("spbleu_") or key == "spbleu_avg":
            continue
        country = key.removeprefix("spbleu_")
        country_rows.append({
            "country": country,
            "spbleu": value,
            "chrfpp": scores.get(f"chrfpp_{country}"),
        })

    try:
        import pandas as pd
        from IPython.display import display

        display(pd.DataFrame([{
            "spbleu_avg": scores["spbleu_avg"],
            "chrfpp_avg": scores["chrfpp_avg"],
            "countries": scores["num_countries"],
            "conversations": scores["num_conversations"],
            "turns": scores["num_turns"],
        }]))
        display(pd.DataFrame(country_rows).sort_values("country").reset_index(drop=True))
    except Exception:
        print(json.dumps({key: scores[key] for key in summary_keys}, indent=2, ensure_ascii=False))
        print(json.dumps(sorted(country_rows, key=lambda row: row["country"]), indent=2, ensure_ascii=False))

## Evaluate on the development split

Note: unlike the official baseline, this doesn't write a CodaBench submission
zip — it stops at local scoring. See `models/infer.py --make-submission-zip`
for the submission-file step.

In [ ]:
development_predictions = generate_prediction_records(
    development_records,
    model,
    tokenizer,
    desc="Generating development predictions",
)
development_predictions_path = write_jsonl(
    development_predictions,
    EVAL_DIR / "development_predictions.jsonl",
    desc="Writing development predictions",
)
development_scores = score_prediction_records(
    development_predictions,
    development_records,
    SCORE_DIR / "development_scores.json",
    desc="Scoring development countries",
)
display_scores(development_scores)